In [9]:

import pandas as pd
import numpy as np

# =========================
# 1. LOAD DATA
# =========================
file_path = r"../../data/austria/raw/time_series_60min_singleindex.csv"

df = pd.read_csv(file_path)

# =========================
# 2. FIX BROKEN COLUMN NAMES
# =========================
# convert u_t_c___t_i_m_e_s_t_a_m_p → utctimestamp
df.columns = [''.join(col.split('_')).lower() for col in df.columns]

# =========================
# 3. RENAME IMPORTANT COLUMNS
# =========================
df = df.rename(columns={
    'utctimestamp': 'timestamp',
    'atloadactualentsoetransparency': 'load_actual',
    'atloadforecastentsoetransparency': 'load_forecast',
    'atpricedayahead': 'price',
    'atsolargenerationactual': 'solar',
    'atwindonshoregenerationactual': 'wind'
})

# =========================
# 4. FIX TIMESTAMP (CRITICAL STEP)
# =========================
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

df = df.set_index('timestamp')

# =========================
# 5. SORT + ENSURE HOURLY CONTINUITY
# =========================
df = df.sort_index()

full_range = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq='H',
    tz='UTC'
)

df = df.reindex(full_range)

# =========================
# 6. HANDLE MISSING VALUES (ENHANCED STRATEGY)
# =========================

# --- SOLAR SPECIAL ---
# Step A: Force solar production to 0 during night hours (approx. 20:00 to 06:00)
# This is a physical constraint to prevent "ghost" production from interpolation or ffill
df.loc[(df.index.hour >= 20) | (df.index.hour <= 6), 'solar'] = 0

# Step B: Linearly interpolate solar values during daytime for small gaps (max. 3h)
# This better reflects the sun's trajectory compared to a simple forward-fill
df['solar'] = df['solar'].interpolate(method='linear', limit=3)

# Step C: Fill remaining solar gaps (e.g., full missing days) with 0
# Ensures no NaNs remain while avoiding the creation of artificial peaks
df['solar'] = df['solar'].fillna(0)

# --- WIND & LOAD ---
# Step D: Use linear interpolation for Wind and Load
# This creates a smooth trend between the last known and next known value,
# effectively using the mean-slope to bridge gaps instead of a frozen constant.
df['wind'] = df['wind'].interpolate(method='linear')
df['load_actual'] = df['load_actual'].interpolate(method='linear')
df['load_forecast'] = df['load_forecast'].interpolate(method='linear')

# Step E: Final Safety Fallback
# In case gaps are at the very beginning or very end of the dataset 
# (where no "next/previous value" exists to interpolate from), we use ffill/bfill.
df = df.ffill().bfill()

# =========================
# 7. REMOVE DUPLICATES (SAFETY)
# =========================
df = df[~df.index.duplicated(keep='first')]

# =========================
# 8. KEEP ONLY AUSTRIA (REDUCE FEATURES)
# =========================
df = df[['load_actual', 'load_forecast', 'solar', 'wind']]

# =========================
# 9. OUTLIER TREATMENT
# =========================
# Uses a mild IQR factor (3.0 instead of 1.5) to preserve natural energy peaks while clipping extreme measurement errors.
def cap_outliers_mild(col):
    Q1 = col.quantile(0.25)
    Q3 = col.quantile(0.75)
    IQR = Q3 - Q1
    # Using 3.0 IQR to ensure we don't smooth out real high-demand or high-generation days
    return np.clip(col, Q1 - 3.0*IQR, Q3 + 3.0*IQR)

# Apply only to numeric data columns, not the timestamp
data_columns = ['load_actual', 'load_forecast', 'solar', 'wind']

for col in data_columns:
    if col in df.columns:
        df[col] = cap_outliers_mild(df[col])

# =========================
# 11. FINAL OUTPUT
# =========================
df_export = df.copy()
if df_export.index.name != 'timestamp':
    df_export.index.name = 'timestamp'

df_export = df_export.reset_index()

output_path = r"../../data/austria/raw/austria_clean_final.csv"
df_export.to_csv(output_path, index=False)

print("Data cleaned successfully")

C:\Users\betti\AppData\Local\Temp\ipykernel_50472\414630352.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_range = pd.date_range(


Data cleaned successfully


In [10]:
print("Shape:", df.info())
print(df.head())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 50401 entries, 2014-12-31 23:00:00+00:00 to 2020-09-30 23:00:00+00:00
Freq: h
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   load_actual    50401 non-null  float64
 1   load_forecast  50401 non-null  float64
 2   solar          50401 non-null  float64
 3   wind           50401 non-null  float64
dtypes: float64(4)
memory usage: 2.9 MB
Shape: None
                           load_actual  load_forecast  solar  wind
2014-12-31 23:00:00+00:00       5946.0         6701.0    0.0  69.0
2015-01-01 00:00:00+00:00       5946.0         6701.0    0.0  69.0
2015-01-01 01:00:00+00:00       5726.0         6593.0    0.0  64.0
2015-01-01 02:00:00+00:00       5347.0         6482.0    0.0  65.0
2015-01-01 03:00:00+00:00       5249.0         6454.0    0.0  64.0
        load_actual  load_forecast         solar          wind
count  50401.000000   50401.000000  50401.000000  50401.